# GCP Secret Engine

In [1]:
%%bash
# Authenticate to GCP (will open browser windows)
gcloud auth application-default login && gcloud auth login


Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=LOz2cfb44wt8UJYVnhqgjzLsK4Ljlz&access_type=offline&code_challenge=CHWlTH87jOZJbEPWo2Afpi9KdwyNvjBHxwWeCUoFkWc&code_challenge_method=S256


Credentials saved to file: [/Users/jose/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "hc-0f488366ab0f42dba5be9a3d890" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.
Your browser has been opened to visit

In [2]:
! open -a Podman\ Desktop

In [3]:
import subprocess, os

# Auto-detect host IP address (DHCP-friendly)
for iface in ["en0", "en1", "en3", "en5"]:
    try:
        ip = subprocess.check_output(["ipconfig", "getifaddr", iface], text=True).strip()
        if ip:
            break
    except Exception:
        ip = None
if not ip:
    ip = "127.0.0.1"

os.environ["VAULT_IP"] = ip
os.environ["VAULT_ADDR"] = f"http://{ip}:8200"
os.environ["VAULT_TOKEN"] = "root"
os.environ["VAULT_PORT"] = "8200"
os.environ["VAULT_KMIP_PORT"] = "5696"

# Get current GCP project
project = "hc-0f488366ab0f42dba5be9a3d890"
os.environ["GCP_PROJECT"] = project

print(f"VAULT_IP:    {ip}")
print(f"VAULT_ADDR:  http://{ip}:8200")
print(f"GCP_PROJECT: {project}")

VAULT_IP:    192.168.1.43
VAULT_ADDR:  http://192.168.1.43:8200
GCP_PROJECT: hc-0f488366ab0f42dba5be9a3d890


### Create a Vault Server in Podman

In [4]:
%%bash
# Change the path to your license file

export VAULT_LICENSE=$(cat vault.hclic)

# Refresh Vault docker image with latest version
podman pull hashicorp/vault-enterprise

podman run -d --rm --name vault-enterprise \
  --cap-add IPC_LOCK \
  -e "VAULT_DEV_ROOT_TOKEN_ID=${VAULT_TOKEN}" \
  -e "VAULT_DEV_LISTEN_ADDRESS=:${VAULT_PORT}" \
  -e "VAULT_LICENSE=${VAULT_LICENSE}" \
  -e "SKIP_SETCAP=true" \
  -p ${VAULT_IP}:${VAULT_KMIP_PORT}:${VAULT_KMIP_PORT} \
  -p ${VAULT_IP}:8200:${VAULT_PORT} \
  hashicorp/vault-enterprise:latest

Trying to pull docker.io/hashicorp/vault-enterprise:latest...
Getting image source signatures
Copying blob sha256:342bbdc3e62e0d2b5da74edb150faa786c17293ed5c593f6a7121bea0c2fddbe
Copying blob sha256:eb42b2effb7d4cf7a1c6f1048801f3819acc8f8f6b4232eefacaab12f1e874d5
Copying blob sha256:593dd0b5c570634bd369264b44c57b503f1fa885eec5760dd385fbe7cf8f9f28
Copying blob sha256:9268c2c682e14c33908ef70b5501cf78da853737da5beb3e693997fb6fafe7c2
Copying blob sha256:34ee9659e1150a325b17f9093ea514389e2cb5df2a2ac51daf8d68bb2dea2530
Copying blob sha256:6c91f4934d2e45338b82071be399cc964755cc1752c0989779a341b76b1dae17
Copying blob sha256:f6a1e3f7f217d9d1436bb88e0d7c79a959469852427f0a4ae42b4e95d7a9de80
Copying config sha256:50676f915ccf3e0dd40471dd3d851b0392477d7fb50b6bfc4be9915a6daf1870
Writing manifest to image destination


50676f915ccf3e0dd40471dd3d851b0392477d7fb50b6bfc4be9915a6daf1870
7aa86514d72bbec99dfcf213926faa19a542dea391108c5b64b8828456630aa4


### Check if Vault is running

In [ ]:
! podman ps

In [5]:
%%bash
(podman ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}' && echo '---' && podman inspect vault-enterprise --format 'HostPortMapping: {{json .NetworkSettings.Ports}}' 2>/dev/null && podman inspect vault-enterprise --format 'ContainerIP: {{.NetworkSettings.IPAddress}}' 2>/dev/null && echo '---' && (ipconfig getifaddr en0 || ipconfig getifaddr en1))

NAMES             STATUS                   PORTS
vault-enterprise  Up 5 minutes (starting)  192.168.1.43:5696->5696/tcp, 192.168.1.43:8200->8200/tcp
---
HostPortMapping: {"5696/tcp":[{"HostIp":"192.168.1.43","HostPort":"5696"}],"8200/tcp":[{"HostIp":"192.168.1.43","HostPort":"8200"}]}
ContainerIP: 10.88.0.7
---
192.168.1.43


## Enable the GCP secrets engine

In [6]:
! vault secrets enable gcp

Success! Enabled the gcp secrets engine at: gcp/


### Create `vault-admin` Service Account
The GCP secrets engine needs a service account with [specific permissions](https://developer.hashicorp.com/vault/docs/secrets/gcp#required-permissions) depending on the credential type:

**For rolesets** (project-level):
```
iam.serviceAccounts.create/delete/get/list/update
iam.serviceAccountKeys.create/delete/get/list
```

**For static accounts** (service-account-level):
```
iam.serviceAccounts.getAccessToken          # access_token secrets
iam.serviceAccountKeys.create/delete/get/list  # service_account_key secrets
```

**For rolesets/static accounts with bindings:**
```
resourcemanager.projects.getIamPolicy/setIamPolicy
```

We assign the following predefined roles to cover all scenarios:
- `roles/iam.serviceAccountAdmin` — manage service accounts (rolesets)
- `roles/iam.serviceAccountKeyAdmin` — manage service account keys (rolesets + rotate-root)
- `roles/resourcemanager.projectIamAdmin` — manage IAM policy bindings
- `roles/iam.serviceAccountTokenCreator` — generate access tokens (static accounts + impersonated accounts)

Additionally, **SA-level self-bindings** are needed for `rotate-root` (vault-admin must manage its own keys).

In [7]:
%%bash
SA_NAME="vault-admin"
SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"

echo "Project:         $GCP_PROJECT"
echo "Service Account: $SA_EMAIL"

# Create service account (skip if already exists)
if gcloud iam service-accounts describe "$SA_EMAIL" &>/dev/null; then
    echo "Service account $SA_NAME already exists, skipping creation."
else
    gcloud iam service-accounts create "$SA_NAME" \
        --display-name="Vault Admin SA" \
        --description="Service account used by Vault GCP secrets engine"
    echo "Service account created."
fi

# Grant required project-level roles
for ROLE in roles/iam.serviceAccountAdmin roles/iam.serviceAccountKeyAdmin \
    roles/resourcemanager.projectIamAdmin roles/iam.serviceAccountTokenCreator; do
    echo "Granting $ROLE..."
    gcloud projects add-iam-policy-binding "$GCP_PROJECT" \
        --member="serviceAccount:${SA_EMAIL}" \
        --role="$ROLE" \
        --condition=None \
        --quiet 2>/dev/null
done

# Grant SA-level self-bindings so vault-admin can manage its own keys (needed for rotate-root)
echo ""
echo "Granting SA-level self-bindings on vault-admin..."
gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
    --member="serviceAccount:${SA_EMAIL}" \
    --role="roles/iam.serviceAccountKeyAdmin" \
    --quiet 2>/dev/null
echo "Granted iam.serviceAccountKeyAdmin on self."

# Delete existing user-managed keys
echo ""
echo "Cleaning up old user-managed keys..."
for KEY_ID in $(gcloud iam service-accounts keys list --iam-account="$SA_EMAIL" \
    --managed-by=user --format="value(name)" 2>/dev/null); do
    gcloud iam service-accounts keys delete "$KEY_ID" \
        --iam-account="$SA_EMAIL" --quiet
    echo "  Deleted key: $KEY_ID"
done

# Create a new key
echo ""
echo "Creating new service account key..."
gcloud iam service-accounts keys create /tmp/vault-admin-key.json \
    --iam-account="$SA_EMAIL"
echo "Key saved to /tmp/vault-admin-key.json"

Project:         hc-0f488366ab0f42dba5be9a3d890
Service Account: vault-admin@hc-0f488366ab0f42dba5be9a3d890.iam.gserviceaccount.com
Service account vault-admin already exists, skipping creation.
Granting roles/iam.serviceAccountAdmin...
bindings:
- members:
  - group:gcpsg-project-hc-0f488366ab0f42dba5be9a3d890-developer@hashicorp.com
  role: organizations/1091910689833/roles/doormat_temp_editor_supplement
- members:
  - serviceAccount:service-884898297574@gcp-sa-bigquerydatatransfer.iam.gserviceaccount.com
  role: roles/bigquerydatatransfer.serviceAgent
- members:
  - serviceAccount:884898297574@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-884898297574@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-884898297574@gcf-admin-robot.iam.gserviceaccount.com
  role: roles/cloudfunctions.serviceAgent
- members:
  - serviceAccount:884898297574@cloudservices.gser

deleted key [ec2b0b388088a32364a32d2c155ffa7fe6cd40bc] for service account [vault-admin@hc-0f488366ab0f42dba5be9a3d890.iam.gserviceaccount.com]


  Deleted key: ec2b0b388088a32364a32d2c155ffa7fe6cd40bc

Creating new service account key...


created key [bd3b4c977f8f932c9b095b607ce1590f3f5656af] of type [json] as [/tmp/vault-admin-key.json] for [vault-admin@hc-0f488366ab0f42dba5be9a3d890.iam.gserviceaccount.com]


Key saved to /tmp/vault-admin-key.json


### Configure Vault GCP Secrets Engine
Pass the `vault-admin` service account key to the GCP secrets engine.

In [11]:
%%bash
# Configure Vault GCP secrets engine with service account credentials
vault write gcp/config \
    credentials=@/tmp/vault-admin-key.json

echo "GCP secrets engine configured."

Success! Data written to: gcp/config
GCP secrets engine configured.


## Rotate Root Credentials

In [12]:
! vault write -f gcp/config/rotate-root

Key               Value
---               -----
private_key_id    5ba518cb2dc1e49245d2972b0bfa87bc3969343a


## Static Accounts
A [static account](https://developer.hashicorp.com/vault/docs/secrets/gcp#static-accounts) maps a Vault path to an **existing** GCP service account.  
Vault can then generate **access tokens** or **service account keys** on demand for that service account.

We will create two static accounts:
1. **`app-token`** — generates OAuth2 **access tokens** (short-lived, auto-expire)
2. **`app-key`** — generates **service account keys** (JSON credentials for JWT signing)

### Create GCP Service Accounts for Static Accounts
We create two service accounts:
- `vault-static-token` — for the access token static account
- `vault-static-key` — for the service account key / JWT static account

In [13]:
%%bash
ADMIN_SA="vault-admin@${GCP_PROJECT}.iam.gserviceaccount.com"

for SA_NAME in vault-static-token vault-static-key; do
    SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"
    echo "=== Service Account: $SA_NAME ==="

    # Create service account (skip if already exists)
    if gcloud iam service-accounts describe "$SA_EMAIL" &>/dev/null; then
        echo "Already exists, skipping creation."
    else
        gcloud iam service-accounts create "$SA_NAME" \
            --display-name="Vault Static SA ($SA_NAME)" \
            --description="Service account managed by Vault static account"
        echo "Created."
    fi

    # Grant Viewer role for testing (so we can verify credentials work)
    gcloud projects add-iam-policy-binding "$GCP_PROJECT" \
        --member="serviceAccount:${SA_EMAIL}" \
        --role="roles/viewer" \
        --condition=None \
        --quiet 2>/dev/null
    echo "Granted roles/viewer."

    # vault-admin needs tokenCreator + keyAdmin on these SAs
    gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
        --member="serviceAccount:${ADMIN_SA}" \
        --role="roles/iam.serviceAccountTokenCreator" \
        --quiet 2>/dev/null
    gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
        --member="serviceAccount:${ADMIN_SA}" \
        --role="roles/iam.serviceAccountKeyAdmin" \
        --quiet 2>/dev/null
    echo "Granted vault-admin tokenCreator + keyAdmin on $SA_NAME"
    echo ""
done

=== Service Account: vault-static-token ===


Already exists, skipping creation.
bindings:
- members:
  - group:gcpsg-project-hc-0f488366ab0f42dba5be9a3d890-developer@hashicorp.com
  role: organizations/1091910689833/roles/doormat_temp_editor_supplement
- members:
  - serviceAccount:service-884898297574@gcp-sa-bigquerydatatransfer.iam.gserviceaccount.com
  role: roles/bigquerydatatransfer.serviceAgent
- members:
  - serviceAccount:884898297574@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-884898297574@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-884898297574@gcf-admin-robot.iam.gserviceaccount.com
  role: roles/cloudfunctions.serviceAgent
- members:
  - serviceAccount:884898297574@cloudservices.gserviceaccount.com
  role: roles/compute.instanceGroupManagerServiceAgent
- members:
  - serviceAccount:service-884898297574@compute-system.iam.gserviceaccount.com
  role: roles/compute.serviceAgent
- memb

### Static Account — Access Tokens (`app-token`)
Configure a static account that generates **OAuth2 access tokens** for `vault-static-token`.  
Tokens are short-lived (default 1 hour) and automatically expire — no need to revoke.

In [14]:
%%bash
SA_EMAIL="vault-static-token@${GCP_PROJECT}.iam.gserviceaccount.com"

vault write gcp/static-account/app-token \
    service_account_email="$SA_EMAIL" \
    secret_type="access_token" \
    token_scopes="https://www.googleapis.com/auth/cloud-platform"

echo ""
echo "Static account created. Reading config:"
vault read gcp/static-account/app-token

Success! Data written to: gcp/static-account/app-token

Static account created. Reading config:
Key                        Value
---                        -----
secret_type                access_token
service_account_email      vault-static-token@hc-0f488366ab0f42dba5be9a3d890.iam.gserviceaccount.com
service_account_project    hc-0f488366ab0f42dba5be9a3d890
token_scopes               [https://www.googleapis.com/auth/cloud-platform]


### Test — Read Access Token

In [15]:
! vault read gcp/static-account/app-token/token

Key                   Value
---                   -----
expires_at_seconds    1776766409
token                 ya29.c.c0AZ4bNpYudRgawv3SYdbJ5ViMpVjBWjQwuBrS56giHAd-DRKyiqWcTxhPy8cuiaUfhVWpjpZmzud3OFXE4ypjEoWDdUEg86ZzTn_Er0XkeRqMGTHCTWhjx1TDilLThVUmEBqnNH38HJWgFxtrouOz_umViiNrK_3ENmRtyNvJ2LAI_bPhnmNdOYxMJXRvjSvKaYFQ9CQqrOAOGzYczJSPUqnTynAcdJs7VEbOucyyJMqqNtuOqXpr9WFr-fhawNF1oZ5hl85fdPmigG875uCpFabLoyR0ryRQLndCwvYchMxrNu64_2yHMJldEoK4NK6bfd4dl5220hh6IRWsK1Y6QicFkFUoQitmg1gUwAjFjzlf4Xd4jQq48v05H385AsWeqO4uuFqagjF8uJgbQ4B8Jla-JRVj0QMeaYxpMYBiwbI31QY8Wyo8zFWsiVn3uZhtQhuahM-1tJzsr-dJVtjXoS5QfX42qJ71oiWdz44qOxujyS_QuIai240hRuBQUgZ3z9ds8eSmyktuYeyV5QkxzFx1vJcrW166VyqOxzXWXs-qQ0dhleVovYicSc81pMmnr3rucMFQmtwV1toV__Y-1u2qOj-zzOVQMz-s4wpm6y6wwYlzR4t3y_B1hafqSqsQpljZ2SwRb_dRuFWeUin_4BzdlRUj8Zr_ziQBo9Vwefd-VqnubZuXXodfFqVZgMluomwJmSiIJSdss_dgRxdqF0ZMBRvosd3e_-5Rydwg6aY4-aJg_WgeFj-XZ0ovb0biM8n4386go2pJRs1zk8WB6-RYxVnlf9oasfWoyJt6t3479bXhrWJmh8S27y7vidOwSBna3Xzo8Fq6a7x1O2krgOUY6RapkxfO7irx22IxFQZepYpz

### Verification — Use Access Token with GCP APIs
Read the access token from Vault and use it to call GCP APIs.

In [16]:
%%bash
echo "=== Static Account Access Token ==="
TOKEN_DATA=$(vault read -format=json gcp/static-account/app-token/token)
ACCESS_TOKEN=$(echo "$TOKEN_DATA" | jq -r '.data.token')
EXPIRES=$(echo "$TOKEN_DATA" | jq -r '.data.token_ttl')

echo "Token (first 20 chars): ${ACCESS_TOKEN:0:20}..."
echo "TTL: ${EXPIRES}s"
echo ""

# Use the token to list GCP projects
echo "=== List Projects (using static account token) ==="
curl -s -H "Authorization: Bearer ${ACCESS_TOKEN}" \
    "https://cloudresourcemanager.googleapis.com/v1/projects" \
    | jq '.projects[]? | {projectId, name, lifecycleState}' 2>/dev/null \
    || echo "No projects or insufficient permissions."

echo ""

# Use the token to list compute regions
echo "=== List Compute Regions ==="
curl -s -H "Authorization: Bearer ${ACCESS_TOKEN}" \
    "https://compute.googleapis.com/compute/v1/projects/${GCP_PROJECT}/regions" \
    | jq '[.items[]?.name] | .[:5]' 2>/dev/null \
    || echo "Unable to list regions."

echo ""

# Verify the token identity
echo "=== Token Info ==="
curl -s "https://oauth2.googleapis.com/tokeninfo?access_token=${ACCESS_TOKEN}" \
    | jq '{email, scope, expires_in}' 2>/dev/null

=== Static Account Access Token ===
Token (first 20 chars): ya29.c.c0AZ4bNpadvBG...
TTL: 3598s

=== List Projects (using static account token) ===
{
  "projectId": "hc-0f488366ab0f42dba5be9a3d890",
  "name": "gcp-engine-test",
  "lifecycleState": "ACTIVE"
}

=== List Compute Regions ===
[
  "africa-south1",
  "asia-east1",
  "asia-east2",
  "asia-northeast1",
  "asia-northeast2"
]

=== Token Info ===
{
  "email": null,
  "scope": "https://www.googleapis.com/auth/cloud-platform",
  "expires_in": "3598"
}


### Static Account — Service Account Keys / JWT (`app-key`)
Configure a static account that generates **service account keys** (JSON) for `vault-static-key`.  
These keys can be used to:
- Authenticate as the service account from external applications
- **Sign JWTs** for service-to-service authentication
- Generate ID tokens for identity-aware applications

> **Note:** Service account keys are long-lived credentials. Vault tracks and can revoke them.

In [17]:
%%bash
SA_EMAIL="vault-static-key@${GCP_PROJECT}.iam.gserviceaccount.com"

vault write gcp/static-account/app-key \
    service_account_email="$SA_EMAIL" \
    secret_type="service_account_key"

echo ""
echo "Static account created. Reading config:"
vault read gcp/static-account/app-key

Success! Data written to: gcp/static-account/app-key

Static account created. Reading config:
Key                        Value
---                        -----
secret_type                service_account_key
service_account_email      vault-static-key@hc-0f488366ab0f42dba5be9a3d890.iam.gserviceaccount.com
service_account_project    hc-0f488366ab0f42dba5be9a3d890


### Test — Read Service Account Key

In [18]:
! vault read gcp/static-account/app-key/key

Key                 Value
---                 -----
lease_id            gcp/static-account/app-key/key/LacBNx9DTS1q575qMAT2hyKJ
lease_duration      768h
lease_renewable     true
key_algorithm       KEY_ALG_RSA_2048
key_type            TYPE_GOOGLE_CREDENTIALS_FILE
private_key_data    ewogICJ0eXBlIjogInNlcnZpY2VfYWNjb3VudCIsCiAgInByb2plY3RfaWQiOiAiaGMtMGY0ODgzNjZhYjBmNDJkYmE1YmU5YTNkODkwIiwKICAicHJpdmF0ZV9rZXlfaWQiOiAiOTgyYThiYjFjZjQ1YmUzYmI1YmRjYzFhNjMzMDg1MjU2OTk1ZTQ5MCIsCiAgInByaXZhdGVfa2V5IjogIi0tLS0tQkVHSU4gUFJJVkFURSBLRVktLS0tLVxuTUlJRXZnSUJBREFOQmdrcWhraUc5dzBCQVFFRkFBU0NCS2d3Z2dTa0FnRUFBb0lCQVFEWGVmNlljV0NhTDUyT1xuTVRDWTFNZGQ3RUxpTXdIUVY5dE5Pb2hqY05YQ2QvaGR6NmVaM0l2SGxhbjd1Zkh5Y2Jkb1dzbE03MzluMDdTWFxuM1cxelE0T1luMEQxMVpmb0dNTDlDT0NjNjczckJDNGQzUkNMbXdRdUxjLzAzeEpkSkR0N2VGT1VyZytFRWo2K1xucnNuQmFGMWtRVWhmWmZXTEUvcFMxbnA2RHRWV1ByUSs3eVJiK052UElyQjlSR3NSRVJCWjliWG84cnlnV0pEelxuOXhBNU9JaHptRGRseDYyd3h4Z1Y5STRKQjFBcEZtMVBjOUdmQmN5MVF0S2psTTFOdFptYkVoQ2dpU0QxTGhCd1xuMk1tMVh4cFE3NW84S0xO

### Verification — Use Service Account Key to Generate a JWT
Read the service account key from Vault, authenticate with it, and use it to call GCP APIs.

In [ ]:
%%bash
set -euo pipefail

echo "=== Static Account Service Account Key ==="

# Try multiple times because newly created service-account keys can take a few seconds
# to become valid for OAuth token exchange in GCP.
MAX_ATTEMPTS=3
ATTEMPT=1
ACTIVATED=0

while [ "$ATTEMPT" -le "$MAX_ATTEMPTS" ]; do
  echo "Attempt ${ATTEMPT}/${MAX_ATTEMPTS}..."

  KEY_DATA=$(vault read -format=json gcp/static-account/app-key/key)

  # Decode key in jq to avoid platform-specific base64 flags and preserve exact JSON.
  echo "$KEY_DATA" | jq -r '.data.private_key_data | @base64d' > /tmp/vault-static-key.json

  echo "Key type:     $(jq -r '.type' /tmp/vault-static-key.json)"
  echo "Client email: $(jq -r '.client_email' /tmp/vault-static-key.json)"
  echo "Key ID:       $(jq -r '.private_key_id' /tmp/vault-static-key.json)"
  echo ""

  if gcloud auth activate-service-account --key-file=/tmp/vault-static-key.json --quiet; then
    ACTIVATED=1
    break
  fi

  echo "Activation failed; waiting for IAM propagation before retry..."
  ATTEMPT=$((ATTEMPT + 1))
  sleep 8
done

if [ "$ACTIVATED" -ne 1 ]; then
  echo "Failed to activate service account after ${MAX_ATTEMPTS} attempts."
  echo "You can inspect the last key file at /tmp/vault-static-key.json"
  exit 1
fi

echo "=== Authenticated Identity ==="
gcloud auth list --filter=status:ACTIVE --format="value(account)"
echo ""

echo "=== List Projects (using service account key) ==="
gcloud projects list --format="table(projectId, name, lifecycleState)" 2>/dev/null | head -10
echo ""

echo "=== Generate Access Token from Key (JWT flow) ==="
# gcloud uses the key to sign a JWT internally and exchanges it for an access token
ACCESS_TOKEN=$(gcloud auth print-access-token)
echo "Access token (first 20 chars): ${ACCESS_TOKEN:0:20}..."
echo ""

echo "=== Token Info ==="
curl -s "https://oauth2.googleapis.com/tokeninfo?access_token=${ACCESS_TOKEN}" \
    | jq '{email, scope, expires_in}' 2>/dev/null

# Clean up
SA_EMAIL=$(jq -r '.client_email' /tmp/vault-static-key.json)
rm -f /tmp/vault-static-key.json
gcloud auth revoke "$SA_EMAIL" --quiet 2>/dev/null || true

# Clean up

In [ ]:
%%bash
echo "=== Deleting Vault rolesets and static accounts ==="
vault delete gcp/static-account/app-token 2>/dev/null && echo "Deleted static account app-token" || true
vault delete gcp/static-account/app-key 2>/dev/null && echo "Deleted static account app-key" || true
vault secrets disable gcp 2>/dev/null && echo "Disabled mount gcp/" || true
echo "Vault resources cleaned up."

In [ ]:
%%bash
echo "=== Deleting GCP Service Accounts ==="
for SA_NAME in vault-admin vault-static-token vault-static-key; do
    SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"
    echo ""
    echo "--- Deleting: $SA_NAME ---"

    # Delete all user-managed keys
    for KEY_ID in $(gcloud iam service-accounts keys list --iam-account="$SA_EMAIL" \
        --managed-by=user --format="value(name)" 2>/dev/null); do
        gcloud iam service-accounts keys delete "$KEY_ID" \
            --iam-account="$SA_EMAIL" --quiet 2>/dev/null
        echo "  Deleted key: $(echo $KEY_ID | rev | cut -d'/' -f1 | rev)"
    done

    # Remove project-level IAM bindings
    for ROLE in roles/iam.serviceAccountAdmin roles/iam.serviceAccountKeyAdmin \
        roles/resourcemanager.projectIamAdmin roles/iam.serviceAccountTokenCreator roles/viewer; do
        gcloud projects remove-iam-policy-binding "$GCP_PROJECT" \
            --member="serviceAccount:${SA_EMAIL}" \
            --role="$ROLE" \
            --quiet 2>/dev/null && echo "  Removed $ROLE" || true
    done

    # Delete the service account
    gcloud iam service-accounts delete "$SA_EMAIL" --quiet 2>/dev/null \
        && echo "  Service account $SA_NAME deleted." \
        || echo "  Service account $SA_NAME not found or already deleted."
done

echo ""

echo "=== GCP cleanup complete ==="

In [ ]:
%%bash
echo "=== Stopping Vault container ==="
podman stop vault-enterprise 2>/dev/null && echo "Vault container stopped." || echo "Vault container not running."
echo "=== All demo resources cleaned up ==="